In [ ]:
import cv2
import numpy as np
import easyocr
import re

reader = easyocr.Reader(
    ['fa'],
    gpu=False)
image_path = "D:\\Plate\\7.jpg"

image = cv2.imread(image_path)

if image is None:
    raise ValueError("Image not found!")


height, width = image.shape[:2]

resized_image = cv2.resize(image,(width * 2, height * 2),interpolation=cv2.INTER_CUBIC)

gray = cv2.cvtColor(resized_image,cv2.COLOR_BGR2GRAY)

clahe = cv2.createCLAHE(clipLimit=3.0,tileGridSize=(8, 8))

clahe_image = clahe.apply(gray)

kernel_sharpen = np.array([[0, -1, 0],[-1, 5, -1],[0, -1, 0]])

sharpened = cv2.filter2D(clahe_image,-1,kernel_sharpen)


adaptive_thresh = cv2.adaptiveThreshold(sharpened,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY,11,2)

kernel = np.ones((2, 2), np.uint8)

morph = cv2.morphologyEx(adaptive_thresh,cv2.MORPH_CLOSE,kernel)

inverted = cv2.bitwise_not(morph)


cv2.imwrite("01_gray.jpg", gray)
cv2.imwrite("02_clahe.jpg", clahe_image)
cv2.imwrite("03_sharpened.jpg", sharpened)
cv2.imwrite("04_threshold.jpg", morph)
cv2.imwrite("05_inverted.jpg", inverted)


ocr_images = [
    ("gray", gray),
    ("clahe", clahe_image),
    ("sharpened", sharpened),
    ("threshold", morph),
    ("inverted", inverted)]

all_results = []

for image_name, img in ocr_images:

    results = reader.readtext(img,detail=1,paragraph=False)

    for result in results:

        bbox = result[0]
        text = result[1]
        confidence = result[2]

        all_results.append({
            "source": image_name,
            "bbox": bbox,
            "text": text,
            "confidence": confidence
        })


if len(all_results) == 0:
    print("No text detected.")
    exit()

best_result = max(all_results,key=lambda x: x["confidence"])


allowed_chars = set(
    "۰۱۲۳۴۵۶۷۸۹"
    "ابپتثجچحخدذرزژسشصضطظعغفقکگلمنوهی")

filtered_text = "".join(
    ch for ch in best_result["text"]
    if ch in allowed_chars)


english_to_persian = str.maketrans(
    "0123456789",
    "۰۱۲۳۴۵۶۷۸۹")

filtered_text = filtered_text.translate(
    english_to_persian)



print("RESULT :")


print(f"Source      : {best_result['source']}")
print(f"Text        : {best_result['text']}")
print(f"Filtered    : {filtered_text}")
print(f"Confidence  : {best_result['confidence']:.4f}")


plate_pattern = r'.*'

if re.match(plate_pattern, filtered_text):
    print("Plate format check: PASSED")
else:
    print("Plate format check: FAILED")